In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import learning_curve
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)

print("Libraries loaded!")

## 11.1 Bias-Variance Tradeoff

In [ ]:
# True function
def true_function(x):
    return np.sin(2 * np.pi * x)

# Generate data
n_samples = 50
X = np.sort(np.random.rand(n_samples))
y_true = true_function(X)
y = y_true + np.random.randn(n_samples) * 0.3

X_test = np.linspace(0, 1, 200)
y_test_true = true_function(X_test)

# Different model complexities
degrees = [1, 3, 9, 15]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, degree in enumerate(degrees):
    # Fit polynomial
    model = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=0.001))
    model.fit(X.reshape(-1, 1), y)
    y_pred = model.predict(X_test.reshape(-1, 1))
    
    # Plot
    axes[idx].scatter(X, y, alpha=0.5, s=30, label='Data')
    axes[idx].plot(X_test, y_test_true, 'g-', linewidth=2, label='True Function')
    axes[idx].plot(X_test, y_pred, 'r-', linewidth=2, label=f'Degree {degree}')
    
    # Compute MSE
    mse = np.mean((y_pred - y_test_true)**2)
    
    if degree == 1:
        axes[idx].set_title(f'High Bias (Underfitting)\nMSE: {mse:.3f}', 
                           fontsize=12, fontweight='bold')
    elif degree == 15:
        axes[idx].set_title(f'High Variance (Overfitting)\nMSE: {mse:.3f}', 
                           fontsize=12, fontweight='bold')
    else:
        axes[idx].set_title(f'Degree {degree} Polynomial\nMSE: {mse:.3f}', 
                           fontsize=12, fontweight='bold')
    
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim(-2, 2)

plt.suptitle('Bias-Variance Tradeoff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11.2 Learning Curves

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import learning_curve

# Generate dataset
X_reg, y_reg = make_regression(n_samples=500, n_features=10, noise=10, random_state=42)

# Compare simple vs complex model
models = [
    ('Simple (Ridge α=10)', Ridge(alpha=10)),
    ('Complex (Ridge α=0.01)', Ridge(alpha=0.01))
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (name, model) in enumerate(models):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_reg, y_reg, cv=5, 
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='neg_mean_squared_error'
    )
    
    train_scores_mean = -np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    val_scores_mean = -np.mean(val_scores, axis=1)
    val_scores_std = np.std(val_scores, axis=1)
    
    axes[idx].plot(train_sizes, train_scores_mean, 'o-', linewidth=2, label='Training Error')
    axes[idx].fill_between(train_sizes, 
                           train_scores_mean - train_scores_std,
                           train_scores_mean + train_scores_std, alpha=0.2)
    
    axes[idx].plot(train_sizes, val_scores_mean, 's-', linewidth=2, label='Validation Error')
    axes[idx].fill_between(train_sizes,
                           val_scores_mean - val_scores_std,
                           val_scores_mean + val_scores_std, alpha=0.2)
    
    axes[idx].set_xlabel('Training Set Size', fontsize=12)
    axes[idx].set_ylabel('MSE', fontsize=12)
    axes[idx].set_title(name, fontsize=13, fontweight='bold')
    axes[idx].legend(fontsize=11)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nLearning Curve Diagnosis:")
print("- High bias: Both errors plateau at high values")
print("- High variance: Large gap between train and validation errors")
print("- More data helps high variance, not high bias")

## 11.3 VC Dimension Illustration

In [ ]:
# Illustrate VC dimension for linear classifiers in 2D
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 2 points - always separable
points_2 = np.array([[0, 0], [1, 1]])
axes[0].scatter(points_2[:1, 0], points_2[:1, 1], c='blue', s=200, marker='o', edgecolors='black')
axes[0].scatter(points_2[1:, 0], points_2[1:, 1], c='red', s=200, marker='s', edgecolors='black')
axes[0].plot([-.5, 1.5], [1.5, -.5], 'k-', linewidth=2)
axes[0].set_title('2 Points: Always Separable', fontsize=12, fontweight='bold')
axes[0].set_xlim(-0.5, 1.5)
axes[0].set_ylim(-0.5, 1.5)
axes[0].grid(True, alpha=0.3)

# 3 points - separable
points_3 = np.array([[0, 0], [1, 0], [0.5, 1]])
colors_3 = ['blue', 'red', 'red']
markers_3 = ['o', 's', 's']
for i in range(3):
    axes[1].scatter(points_3[i, 0], points_3[i, 1], c=colors_3[i], 
                   s=200, marker=markers_3[i], edgecolors='black')
axes[1].plot([-.2, .3], [.5, -.5], 'k-', linewidth=2)
axes[1].set_title('3 Points: Separable', fontsize=12, fontweight='bold')
axes[1].set_xlim(-0.3, 1.3)
axes[1].set_ylim(-0.3, 1.3)
axes[1].grid(True, alpha=0.3)

# 4 points - XOR not separable
points_4 = np.array([[0, 0], [1, 0], [0, 1], [1, 1]])
colors_4 = ['blue', 'red', 'red', 'blue']
markers_4 = ['o', 's', 's', 'o']
for i in range(4):
    axes[2].scatter(points_4[i, 0], points_4[i, 1], c=colors_4[i], 
                   s=200, marker=markers_4[i], edgecolors='black')
axes[2].text(0.5, 0.5, 'NOT\nLINEARLY\nSEPARABLE', 
            ha='center', va='center', fontsize=11, fontweight='bold', color='red')
axes[2].set_title('4 Points (XOR): Not Separable', fontsize=12, fontweight='bold')
axes[2].set_xlim(-0.3, 1.3)
axes[2].set_ylim(-0.3, 1.3)
axes[2].grid(True, alpha=0.3)

plt.suptitle('VC Dimension of Linear Classifiers in 2D = 3', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nVC Dimension:")
print("- Maximum number of points that can be shattered (all labelings separable)")
print("- Linear classifier in d dimensions: VC dim = d + 1")
print("- Higher VC dim → more capacity → risk of overfitting")

## 11.4 Generalization Bounds

In [ ]:
# Visualize generalization bound
m = np.logspace(2, 4, 100)  # Training set sizes
d_values = [2, 5, 10, 20]  # VC dimensions

plt.figure(figsize=(12, 7))

for d in d_values:
    # Simplified VC bound: O(sqrt(d log m / m))
    bound = np.sqrt((d * np.log(m)) / m)
    plt.plot(m, bound, linewidth=2, label=f'VC dim = {d}')

plt.xscale('log')
plt.xlabel('Training Set Size (m)', fontsize=12)
plt.ylabel('Generalization Error Bound', fontsize=12)
plt.title('VC Generalization Bound vs Training Set Size', 
         fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("- Error bound decreases as O(1/√m)")
print("- Higher complexity (VC dim) requires more data")
print("- Theoretical worst-case bound (often loose in practice)")

## Key Takeaways

### 1. **Bias-Variance Decomposition**
- **Bias**: Error from wrong assumptions (underfitting)
- **Variance**: Error from sensitivity to training data (overfitting)
- Tradeoff: Can't minimize both simultaneously

### 2. **Learning Curves**
- **High bias**: Train/val errors converge at high value
  - Solution: More features, complex model
- **High variance**: Large gap between train/val errors
  - Solution: More data, regularization, simpler model

### 3. **VC Dimension**
- Measure of model complexity
- Linear classifier in d dimensions: VC = d+1
- Higher VC → can fit more patterns → needs more data

### 4. **Generalization Bounds**
- Training error ≠ test error
- Gap depends on: model complexity, training size
- Regularization reduces effective complexity

### 5. **Practical Guidelines**
- Start simple, increase complexity if needed
- Use cross-validation
- Regularization is almost always helpful
- More data helps variance, not bias

## References

1. CS229 Lecture Notes on Learning Theory
2. "The Elements of Statistical Learning" - Hastie et al.
3. "Understanding Machine Learning" - Shalev-Shwartz & Ben-David

---

**Next**: [Lecture 12: ML Strategy](12_ml_strategy.ipynb)